# GAN Example 4 (PyTorch) — Least Squares GAN (LSGAN)

PyTorch version. Same DCGAN architecture as `gan_mnist_pytorch.ipynb`, but the loss is
**mean-squared error** instead of binary cross-entropy:

$$\mathcal{L}_D = \tfrac12\mathbb{E}[(D(x)-1)^2] + \tfrac12\mathbb{E}[D(G(z))^2], \qquad \mathcal{L}_G = \tfrac12\mathbb{E}[(D(G(z))-1)^2].$$

MSE doesn't saturate the way cross-entropy does, so even confidently-rejected fakes keep
receiving a useful gradient — often giving smoother training. The discriminator outputs a
raw number (no sigmoid).

> Needs `torch` + `torchvision`.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

In [ ]:
NOISE_DIM = 100
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform),
                          batch_size=256, shuffle=True, drop_last=True)

## Models (same as the DCGAN)

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(NOISE_DIM, 7*7*256, bias=False),
                                nn.BatchNorm1d(7*7*256), nn.LeakyReLU(0.2, True))
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 5, 1, 2, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.ConvTranspose2d(128, 64, 5, 2, 2, output_padding=1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),
            nn.ConvTranspose2d(64, 1, 5, 2, 2, output_padding=1, bias=False), nn.Tanh())
    def forward(self, z):
        return self.deconv(self.fc(z).view(-1, 256, 7, 7))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 5, 2, 2), nn.LeakyReLU(0.2, True), nn.Dropout(0.3),
            nn.Conv2d(64, 128, 5, 2, 2), nn.LeakyReLU(0.2, True), nn.Dropout(0.3),
            nn.Flatten(), nn.Linear(128*7*7, 1))   # linear output
    def forward(self, x):
        return self.net(x)

G = Generator().to(device)
D = Discriminator().to(device)
mse = nn.MSELoss()
g_opt = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
seed = torch.randn(16, NOISE_DIM, device=device)

## Train (MSE losses)

In [ ]:
def plot_samples(epoch=None):
    G.eval()
    with torch.no_grad():
        imgs = G(seed).cpu() * 0.5 + 0.5
    G.train()
    plt.figure(figsize=(4, 4))
    plt.imshow(make_grid(imgs, nrow=4).permute(1, 2, 0).numpy(), cmap='gray'); plt.axis('off')
    if epoch is not None: plt.title(f'Epoch {epoch}')
    plt.show()

EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    for real, _ in train_loader:
        real = real.to(device); bsz = real.size(0)
        ones, zeros = torch.ones(bsz, 1, device=device), torch.zeros(bsz, 1, device=device)
        # Discriminator: regress real->1, fake->0
        fake = G(torch.randn(bsz, NOISE_DIM, device=device))
        d_loss = 0.5 * (mse(D(real), ones) + mse(D(fake.detach()), zeros))
        d_opt.zero_grad(); d_loss.backward(); d_opt.step()
        # Generator: regress fake->1
        g_loss = 0.5 * mse(D(fake), ones)
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()
    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={g_loss.item():.3f}  disc_loss={d_loss.item():.3f}')
    if epoch % 5 == 0 or epoch == 1:
        plot_samples(epoch)

### Things to try

- Compare with the BCE DCGAN (`gan_mnist_pytorch.ipynb`).
- Vary the regression targets per the LSGAN paper.
- Combine with the conditional setup.
- Apply the MSE loss to the 2D toy problem to watch its effect on mode coverage.